In [1]:
# Import the library
import pandas as pd

In [2]:
# Zones with prices
price_zones = ["PT", "ES", "FR", "BE", "NL", "DE_LU", "AT", "CH", "IT_NORD", "PL", "CZ", "SK", "HU",
               "SI", "HR", "RO", "BG", "GR", "DK_1", "SE_3", "NO_1", "FI", "EE", "LV", "LT", "IE_SEM"]

# Join the prices of all zones in one table, one column per zone
columns = []
for zone in price_zones:
    table = pd.read_csv("../data/raw/entsoe/prices_" + zone + ".csv", index_col="time_utc")
    columns.append(table["price"].rename(zone))
prices = pd.concat(columns, axis=1).sort_index()
print(prices.shape)
prices.tail()

(68111, 26)


,PT,ES,FR,BE,NL,DE_LU,AT,CH,IT_NORD,PL,...,BG,GR,DK_1,SE_3,NO_1,FI,EE,LV,LT,IE_SEM
time_utc,,,,,,,,,,,,,,,,,,,,,
2026-10-08 17:00:00+00:00,176.4700,176.4700,NaN,261.8325,NaN,NaN,224.8600,227.59,NaN,172.5775,...,254.3600,254.3600,152.0900,136.045,139.3550,96.0350,104.5925,162.7075,162.7075,NaN
2026-10-08 18:00:00+00:00,198.8025,198.8025,NaN,221.4575,NaN,NaN,224.3675,219.89,NaN,129.5100,...,240.7475,240.7475,140.2700,121.495,132.2950,80.7975,84.3975,148.8525,148.8525,NaN
2026-10-08 19:00:00+00:00,186.6325,186.6325,NaN,199.4875,NaN,NaN,186.7825,212.09,NaN,111.2425,...,245.8425,245.8425,129.0000,99.210,127.2375,65.4575,65.4950,121.9300,121.9300,NaN
2026-10-08 20:00:00+00:00,177.6750,177.6750,NaN,184.6900,NaN,NaN,185.2125,212.09,NaN,122.0875,...,232.2850,232.2850,114.1300,36.030,118.3450,34.8450,34.8475,36.8350,36.8350,NaN
2026-10-08 21:00:00+00:00,159.6850,159.6850,NaN,175.5925,NaN,NaN,160.4700,191.79,NaN,108.1050,...,230.4175,230.4175,97.1525,27.190,114.9025,27.1900,27.1900,27.1900,27.1900,NaN


In [3]:
# Count the missing hours of each zone
prices.isna().sum()

PT           6
ES           6
FR          30
BE           6
NL          30
DE_LU       30
AT           6
CH           7
IT_NORD     30
PL           6
CZ           6
SK           6
HU           6
SI           6
HR           6
RO           6
BG           6
GR           6
DK_1         6
SE_3         6
NO_1         6
FI           6
EE           6
LV           6
LT           6
IE_SEM     348
dtype: int64

In [4]:
# Zones with production
production_zones = ["PT", "ES", "FR", "DE_LU", "IT_NORD", "NL", "BE", "PL", "AT",
                    "DK_1", "FI", "GR", "HU", "RO", "CZ"]

# Join solar, wind and consumption of all zones in one table, three columns per zone
columns = []
for zone in production_zones:
    table = pd.read_csv("../data/raw/entsoe/production_" + zone + ".csv", index_col="time_utc")
    for column in ["solar", "wind", "load"]:
        columns.append(table[column].rename(zone + "_" + column))
production = pd.concat(columns, axis=1).sort_index()
print(production.shape)
production.tail()

(68063, 45)


,PT_solar,PT_wind,PT_load,ES_solar,ES_wind,ES_load,FR_solar,FR_wind,FR_load,DE_LU_solar,...,GR_load,HU_solar,HU_wind,HU_load,RO_solar,RO_wind,RO_load,CZ_solar,CZ_wind,CZ_load
time_utc,,,,,,,,,,,,,,,,,,,,,
2026-10-06 17:00:00+00:00,288.5,704.4,6888.3,1181.0,2676.0,30631.0,439.4475,2595.1100,49371.1225,0.43600,...,5997.00,28.142520,39.251242,5916.48075,0.0,103.50,6908.25,16.19,29.4875,7384.6825
2026-10-06 18:00:00+00:00,11.7,1111.3,7480.6,65.0,2515.0,30853.0,219.3125,3536.4950,46711.2075,0.23125,...,5642.00,10.424582,29.068150,5654.68700,0.0,143.75,6536.00,0.00,30.3000,7193.2875
2026-10-06 19:00:00+00:00,9.9,1247.7,7361.1,44.0,2498.0,30033.0,0.0000,4622.4600,43144.8150,0.16275,...,5121.75,4.754692,27.989728,5286.87025,0.0,158.50,5952.00,0.00,39.2950,6759.9250
2026-10-06 20:00:00+00:00,9.9,1237.4,6844.9,45.0,2409.0,27771.0,0.0000,5095.7525,42892.7175,0.11650,...,4649.75,2.827595,27.322927,5089.59300,0.0,189.50,5475.25,0.00,53.0525,6421.3750
2026-10-06 21:00:00+00:00,6.7,1164.0,6275.1,44.0,2337.0,25645.0,0.0000,5070.6625,41993.6375,0.10200,...,4389.25,5.488783,37.390890,4906.38600,0.0,210.25,5155.50,0.00,54.0675,6127.5400


In [5]:
# Average solar production per year in each zone, to find strange values
solar_columns = [zone + "_solar" for zone in production_zones]
production.groupby(production.index.str[:4])[solar_columns].mean().round(0)

,PT_solar,ES_solar,FR_solar,DE_LU_solar,IT_NORD_solar,NL_solar,BE_solar,PL_solar,AT_solar,DK_1_solar,FI_solar,GR_solar,HU_solar,RO_solar,CZ_solar
time_utc,,,,,,,,,,,,,,,
2018,0.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,NaN,0.0,0.0
2019,121.0,1647.0,1304.0,4775.0,803.0,8.0,403.0,0.0,76.0,81.0,0.0,412.0,78.0,157.0,258.0
2020,144.0,2281.0,1417.0,5230.0,814.0,24.0,485.0,276.0,92.0,97.0,0.0,453.0,181.0,153.0,246.0
2021,196.0,2894.0,1560.0,5299.0,821.0,47.0,534.0,527.0,96.0,115.0,0.0,542.0,274.0,148.0,249.0
2022,289.0,3549.0,2051.0,6391.0,884.0,58.0,733.0,1061.0,111.0,164.0,0.0,716.0,352.0,156.0,270.0
2023,410.0,4616.0,2459.0,6370.0,937.0,62.0,821.0,1510.0,268.0,267.0,97.0,986.0,502.0,167.0,317.0
2024,559.0,5387.0,2655.0,7223.0,1100.0,55.0,948.0,1975.0,666.0,299.0,137.0,1294.0,647.0,238.0,445.0
2025,697.0,6004.0,3456.0,8472.0,1437.0,57.0,1164.0,2195.0,647.0,372.0,150.0,1312.0,778.0,310.0,537.0
2026,835.0,7913.0,4663.0,11668.0,1731.0,64.0,1552.0,3085.0,914.0,510.0,240.0,1185.0,991.0,606.0,710.0


In [6]:
# Save the two tables, prices with 2 decimals and production in whole MW
prices.to_csv("../data/europe_prices.csv", float_format="%.2f")
production.to_csv("../data/europe_production.csv", float_format="%.0f")